In [1]:
import pandas as pd
spotify=pd.read_csv('dataset.csv')
print(spotify.head())

   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3           3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4           4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Comedy          73       230666     False   
1            Ghost - Acoustic          55       149610     False   
2              To Begin Again          57       210826     False   


In [2]:
print(spotify.isnull().sum())

Unnamed: 0          0
track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64


In [3]:
print(spotify.dtypes)

Unnamed: 0            int64
track_id             object
artists              object
album_name           object
track_name           object
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre          object
dtype: object


In [4]:
from sklearn.preprocessing import LabelEncoder
spotify_clean=spotify.dropna(subset=['artists', 'track_genre']).copy()
spotify_clean['text_info']=spotify_clean['artists'].fillna('')+' '+spotify_clean['album_name'].fillna('')+' '+spotify_clean['track_name'].fillna('')
spotify_clean['explicit']=spotify_clean['explicit'].astype(int)
num_features=[
    'popularity',
    'duration_ms',
    'explicit',
    'danceability',
    'energy',
    'key',
    'loudness',
    'mode',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo',
    'time_signature',
]
cat_features='text_info'
X=spotify_clean[num_features+[cat_features]]
le=LabelEncoder()
y=le.fit_transform(spotify_clean['track_genre'])

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
preprocessor=ColumnTransformer(
    transformers=[
        ('num','passthrough',num_features),
        ('cat',TfidfVectorizer(max_features=4500,stop_words='english',ngram_range=(1,2)),cat_features),
    ]
)
my_pipeline=Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        (
            'classifier',
            XGBClassifier(
                n_estimators=400,
                learning_rate=0.07,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                tree_method='hist',
                device='cuda',
                random_state=0,
                n_jobs=-1,
            ),
        ),
    ]
)

In [8]:
from sklearn.model_selection import train_test_split
X_train ,X_valid, y_train,y_valid= train_test_split(X,y,test_size=0.2,random_state=0,stratify=y)
my_pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [9]:

accuracy=my_pipeline.score(X_valid,y_valid)
print(f"Accuracy : {accuracy * 100:.2f}%")

c:\Users\LENOVO\anaconda3\Lib\site-packages\xgboost\core.py:553: UserWarning: [14:40:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Accuracy : 48.30%
